# Install Dependencies

In [1]:
!pip install langchain langchain_community faiss-cpu langchain_huggingface

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 101.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 548.1/548.1 kB 51.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 9.1 MB/s eta 0:00:00
  Attempting uninstall: requests
    Found existing installation: requests 2.32.4
    Uninstalling requests-2.32.4:
      Successfully uninstalled requests-2.32.4
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.3.1
    Uninstalling langchain-core-1.3.1:
      Successfully uninstalled langchain-core-1.3.1
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is i

In [5]:
!pip install langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 6.0 MB/s eta 0:00:00


In [77]:
!pip install docx2txt

In [57]:
!pip install rank_bm25

In [2]:
!pip install pymupdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.0/25.0 MB 90.1 MB/s eta 0:00:00


# Model Setup With **Groq**

In [3]:
from google.colab import userdata
API_KEY=userdata.get('GROQ_API_KEY')

In [6]:
from langchain_groq import ChatGroq
Model = ChatGroq(
    model="openai/gpt-oss-20b",
    api_key=API_KEY,
    temperature=0.2
)

In [7]:
Model.invoke("Who is the prime minsiter of pakistan?").content

'**Current Prime Minister (as of the latest information available up to June\u202f2024)**  \n- **Anwaar‑ul‑Haq Kakar** – He is serving as the caretaker Prime Minister of Pakistan, appointed in August\u202f2023 to oversee the country until the next general elections.\n\n**Background**  \n- The previous elected Prime Minister, **Shehbaz Sharif**, served from April\u202f2022 until his resignation in August\u202f2023.  \n- After Sharif’s resignation, the Election Commission of Pakistan appointed Anwaar‑ul‑Haq Kakar, a former federal minister and senior politician, as the caretaker head of government.  \n- The caretaker government’s mandate is to conduct free and fair elections and maintain day‑to‑day governance until a new elected government is formed.\n\nIf you’re looking for the name of the *elected* Prime Minister after the upcoming elections, that will be determined once the results are announced.'

# Setup the documnt Loaders

In [8]:
from langchain_community.document_loaders import PyMuPDFLoader,Docx2txtLoader,TextLoader

In [124]:
def load_documents(file_path:str):

  extension = file_path.split(".")[-1]
  print("Captured Extension is:", extension)

  if extension == "pdf":
    loader = PyMuPDFLoader(file_path=file_path)
  elif extension == "docx":
    loader = Docx2txtLoader(file_path=file_path)
  elif extension == "txt":
    loader = TextLoader(file_path=file_path,encoding='utf-8')
  else:
    raise ValueError(
            f"Unsupported file type: {extension}"
        )
  return loader.load()

In [168]:
path="/content/9.-Nineteen-Eighty-Four-Author-George-Orwell (2).pdf"
res = load_documents(file_path=path)


Captured Extension is: pdf


# Preprocess the data into chunks and overlapping

In [169]:
from langchain_text_splitters import RecursiveCharacterTextSplitter


def split_documents(documents):
  text_splitter = RecursiveCharacterTextSplitter(
      chunk_size=1200,
    chunk_overlap=250
  )
  chunks = text_splitter.split_documents(documents)
  return chunks



In [170]:
chunks = split_documents(documents=res)
print(len(chunks))

664


#Loading the embedding model from huggingface i will use lightweight model from huggingfacce.

In [156]:
from langchain_huggingface import HuggingFaceEmbeddings

Embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [157]:
print("Embedding dimnesions are:",len(Embeddings.embed_query("Im prime minister of pakistan")))

Embedding dimnesions are: 384


# Convert chunks to embeddings and than store it into database.

In [158]:
from langchain_community.vectorstores import FAISS

In [159]:
vector_database = FAISS.from_documents(
    documents=chunks,
    embedding=Embeddings
)

In [160]:
retreiver = vector_database.as_retriever(search_kwargs={"k":3}, search_type="similarity")

In [163]:
docs = retreiver.invoke("Frameworks and Technologies Used.")

In [164]:
print(len(docs))

3


In [165]:
for index, value in enumerate(docs):
    print(f"Doc {index}:")
    print(value.page_content[:500])

    print(f"Metadata:")
    print(value.metadata)

Doc 0:
TABLE OF CONTENTS



	CHAPTER	PAGE



	CHAPTER 1: INTRODUCTION	1

		Introduction	2

		Motivation for Developed System	2

		Problem Statement	2

		Developed System Objective	3

		Developed System	3

		Scope of Developed System	4

		Tools & Technology for Developed System	4

		Milestones	4

		Summary	5

	CHAPTER 2: BACKGROUND AND EXISTING SYSTEM	6

		Introduction	7

		Existing Systems	7

		Modules of Developed System	10

		Flow Diagram of Developed System	11

		Methodology used for Developed Syste
Metadata:
{'source': '/content/Fyp-Document001.docx'}
Doc 1:
Poor Interface:



Research prototypes typically have basic, text-heavy interfaces designed for developers, not drivers. They lack intuitive controls and visual feedback, making them difficult to use in real driving conditions.

	No Commercial Viability:



These research prototypes focus on algorithm development rather than user-ready products. They lack the polish, documentation, and support required for commercial deployment

In [167]:
target = "Frameworks and Technologies Used."

matches = []

for chunk in chunks:
    text = chunk.page_content.lower()

    if target.lower() in text:
        matches.append(chunk)

print("Matches found:", len(matches))

for doc in matches:
    print("\nPage:", doc.metadata.get("page"))
    print(doc.page_content[:1000])
    print("=" * 80)

Matches found: 0


In [137]:
from langchain_community.retrievers import BM25Retriever
from langchain_classic.retrievers import EnsembleRetriever

In [139]:
BM25 = BM25Retriever.from_documents(
    chunks
)
BM25.k=3

In [140]:
BM25.invoke("Yes, I consider myself superior")

[Document(metadata={'producer': '3-Heights™ PDF Optimization Shell 6.3.1.5 (http://www.pdf-tools.com)', 'creator': 'Acrobat PDFMaker 22 for Word(Foxit Advanced PDF Editor)', 'creationdate': '2022-03-30T12:31:02+01:00', 'source': '/content/9.-Nineteen-Eighty-Four-Author-George-Orwell (2).pdf', 'file_path': '/content/9.-Nineteen-Eighty-Four-Author-George-Orwell (2).pdf', 'total_pages': 174, 'format': 'PDF 1.6', 'title': 'Nineteen Eighty-Four', 'author': 'George Orwell', 'subject': '', 'keywords': 'https://www.globalgreyebooks.com/nineteen-eighty-four-ebook.html', 'moddate': '2023-10-10T19:34:06+00:00', 'trapped': '', 'modDate': 'D:20231010193406Z', 'creationDate': "D:20220330123102+01'00'", 'page': 150}, page_content='manner changed and he said more harshly: ‘And you consider yourself morally superior to \nus, with our lies and our cruelty?’ \n‘Yes, I consider myself superior.’ \nO’Brien did not speak. Two other voices were speaking. After a moment Winston recognized \none of them as his

In [149]:
Ensemble_Retreiver = EnsembleRetriever(
    retrievers=[BM25,retreiver],
    weights=[0.6, 0.4]
)


In [152]:
Ensemble_Retreiver.invoke("Yes! i considerd myself superior")

[Document(metadata={'producer': '3-Heights™ PDF Optimization Shell 6.3.1.5 (http://www.pdf-tools.com)', 'creator': 'Acrobat PDFMaker 22 for Word(Foxit Advanced PDF Editor)', 'creationdate': '2022-03-30T12:31:02+01:00', 'source': '/content/9.-Nineteen-Eighty-Four-Author-George-Orwell (2).pdf', 'file_path': '/content/9.-Nineteen-Eighty-Four-Author-George-Orwell (2).pdf', 'total_pages': 174, 'format': 'PDF 1.6', 'title': 'Nineteen Eighty-Four', 'author': 'George Orwell', 'subject': '', 'keywords': 'https://www.globalgreyebooks.com/nineteen-eighty-four-ebook.html', 'moddate': '2023-10-10T19:34:06+00:00', 'trapped': '', 'modDate': 'D:20231010193406Z', 'creationDate': "D:20220330123102+01'00'", 'page': 150}, page_content='manner changed and he said more harshly: ‘And you consider yourself morally superior to \nus, with our lies and our cruelty?’ \n‘Yes, I consider myself superior.’ \nO’Brien did not speak. Two other voices were speaking. After a moment Winston recognized \none of them as his